# E(2)-Equivariant ResNet on Rotated MNIST

This notebook demonstrates training an E(2)-equivariant ResNet on the Rotated MNIST dataset.
The model is equivariant to rotations and reflections, making it naturally suited for
recognizing digits regardless of their orientation.

## Key Concepts
- **E(2) Equivariance**: The model's output transforms predictably under 2D rotations and reflections
- **Steerable CNNs**: Use group-equivariant convolutions that share weights across transformations
- **Rotation Invariant Classification**: Final layer produces rotation-invariant features

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Dataset: Rotated MNIST

We create a Rotated MNIST dataset by applying random rotations to standard MNIST digits.
This tests the model's ability to recognize digits regardless of orientation.

In [ ]:
class RotatedMNIST(Dataset):
    """MNIST dataset with random rotations applied to each image."""
    
    def __init__(self, root='./data', train=True, download=True, 
                 rotation_range=180, fixed_rotation=None):
        """
        Args:
            root: Data directory
            train: If True, use training set
            download: If True, download dataset
            rotation_range: Maximum rotation angle (symmetric around 0)
            fixed_rotation: If set, apply this fixed rotation to all images
        """
        self.mnist = torchvision.datasets.MNIST(
            root=root, train=train, download=download
        )
        self.rotation_range = rotation_range
        self.fixed_rotation = fixed_rotation
        
        # Normalize transform
        self.normalize = transforms.Normalize((0.1307,), (0.3081,))
    
    def __len__(self):
        return len(self.mnist)
    
    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        
        # Apply rotation
        if self.fixed_rotation is not None:
            angle = self.fixed_rotation
        else:
            angle = random.uniform(-self.rotation_range, self.rotation_range)
        
        image = transforms.functional.rotate(image, angle)
        
        # Convert to tensor and normalize
        image = transforms.functional.to_tensor(image)
        image = self.normalize(image)
        
        # Pad to 32x32 for easier processing
        image = nn.functional.pad(image, (2, 2, 2, 2))
        
        return image, label


# Create datasets
train_dataset = RotatedMNIST(train=True, download=True, rotation_range=180)
test_dataset = RotatedMNIST(train=False, download=True, rotation_range=180)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

In [ ]:
# Visualize some rotated samples
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

for i in range(16):
    image, label = train_dataset[i]
    ax = axes[i // 8, i % 8]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')

plt.suptitle('Rotated MNIST Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Model: E(2)-Equivariant ResNet

We use a smaller version of E2-ResNet suitable for MNIST-sized images.

In [ ]:
from examples.e2resnet import E2ResNet, BasicBlock

def create_e2resnet_mnist(N=8, flip=True, num_classes=10):
    """
    Create a small E2-ResNet for MNIST (32x32 grayscale images).
    
    Args:
        N: Number of discrete rotations (8 = 45 degree increments)
        flip: If True, also equivariant to reflections
        num_classes: Number of output classes
    """
    model = E2ResNet(
        block=BasicBlock,
        layers=[2, 2, 2, 2],  # ResNet-18 style
        num_classes=num_classes,
        N=N,
        flip=flip,
        restrict=2,  # Progressively reduce equivariance
        in_channels=1,  # Grayscale
        base_width=32,  # Smaller width for MNIST
    )
    return model


# Create model
model = create_e2resnet_mnist(N=8, flip=True)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 3. Training Setup

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
WEIGHT_DECAY = 1e-4

# Data loaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})
    
    return total_loss / total, 100. * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / total, 100. * correct / total

## 4. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'test_loss': [], 'test_acc': []
}

best_acc = 0

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch {epoch + 1}/{NUM_EPOCHS}')
    print('-' * 40)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Evaluate
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')
    
    # Save best model
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'e2resnet_rotmnist_best.pth')
        print(f'Saved best model with accuracy: {best_acc:.2f}%')

print(f'\nBest Test Accuracy: {best_acc:.2f}%')

## 5. Visualize Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['test_loss'], label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['test_acc'], label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Test Rotation Invariance

A key property of E(2)-equivariant networks is rotation invariance.
Let's verify this by testing on images rotated at specific angles.

In [ ]:
def test_rotation_invariance(model, test_angles=[0, 45, 90, 135, 180, 225, 270, 315]):
    """Test model accuracy at different rotation angles."""
    model.eval()
    results = {}
    
    for angle in test_angles:
        # Create dataset with fixed rotation
        dataset = RotatedMNIST(train=False, fixed_rotation=angle)
        loader = DataLoader(dataset, batch_size=128, shuffle=False)
        
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        accuracy = 100. * correct / total
        results[angle] = accuracy
        print(f'Rotation {angle:3d}°: Accuracy = {accuracy:.2f}%')
    
    return results


print('Testing Rotation Invariance:')
print('=' * 40)
rotation_results = test_rotation_invariance(model)

In [ ]:
# Plot rotation invariance results
angles = list(rotation_results.keys())
accuracies = list(rotation_results.values())

plt.figure(figsize=(10, 5))
plt.bar(range(len(angles)), accuracies, tick_label=[f'{a}°' for a in angles])
plt.axhline(y=np.mean(accuracies), color='r', linestyle='--', label=f'Mean: {np.mean(accuracies):.2f}%')
plt.xlabel('Rotation Angle')
plt.ylabel('Accuracy (%)')
plt.title('E(2)-ResNet: Rotation Invariance Test')
plt.legend()
plt.ylim([min(accuracies) - 5, 100])
plt.grid(axis='y')
plt.show()

print(f'\nAccuracy Range: {min(accuracies):.2f}% - {max(accuracies):.2f}%')
print(f'Standard Deviation: {np.std(accuracies):.2f}%')

## 7. Compare with Standard CNN

Let's compare the E(2)-ResNet with a standard (non-equivariant) ResNet
to see the benefits of equivariance.

In [ ]:
# Create a standard ResNet for comparison
from torchvision.models import resnet18

def create_standard_resnet(num_classes=10):
    """Create a standard ResNet-18 for MNIST."""
    model = resnet18(weights=None)
    # Modify for grayscale input
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    # Modify for 10 classes
    model.fc = nn.Linear(512, num_classes)
    return model


# Train standard ResNet (abbreviated for notebook)
print('Note: For a fair comparison, train standard ResNet with the same settings.')
print('The E(2)-equivariant model should show more consistent accuracy across rotations.')

## Summary

This notebook demonstrated:

1. **E(2)-Equivariant ResNet** for rotation-invariant image classification
2. **Rotated MNIST** dataset for testing rotation invariance
3. **Training pipeline** with standard PyTorch practices
4. **Rotation invariance testing** showing consistent accuracy across angles

### Key Advantages of E(2)-Equivariance:
- **No data augmentation needed** for rotation invariance
- **Consistent performance** across all rotation angles
- **Better generalization** to unseen orientations
- **Parameter efficiency** through weight sharing across transformations